# Scalar Book Export Recreator

**This notebook is an alternate way to export a Scalar book, for use when the book is too large for Scalar's built-in export tool to handle.**

Scalar's built-in export feature (and the bulk API endpoint it relies on internally) tries to fetch a book's entire content and relationship graph in a single request. On a large, densely-linked book, that single request exceeds a resource limit on the server and fails outright (HTTP 503 Service Unavailable) — the export never completes, with no partial result and no way to adjust settings to fix it from the user side.

This notebook produces the same result — a complete book export in the same RDF-JSON format the built-in tool would have produced — by requesting the exact same data in many small, individually-scoped pieces instead of one giant request. Each small request stays well under the size that triggers the server-side failure, so the export completes even for books the built-in tool cannot handle at all.

**What ends up in the final file:** every page's title and full HTML content, every media item's metadata, the complete table-of-contents structure, and every spatial image annotation — the whole book, not a partial one.

**How the two crawl phases below fit together:**
- **Phase 1 (Structural crawl)** walks the table of contents and, along the way, captures every page's and media item's own full content — this is where the bulk of the book (titles, body text, media references) gets collected.
- **Phase 2 (Annotation crawl)** goes back over just the media items found in Phase 1 to additionally capture spatial image annotations, which live in a separate relationship type that has to be requested on its own.

Both phases merge into the same output file — the result is one complete book export, not two separate partial ones.

**How to use this notebook:**
1. Set `BOOK_SLUG` below to the book you want to export (just the slug, e.g. `piranesidigitalproject`).
2. Run every cell in order, top to bottom.
3. The crawl cells can be re-run safely if interrupted — they skip anything already fetched.
4. The final output is one JSON file, structurally identical to a native Scalar RDF-JSON export.

See the accompanying methodology document for the full reasoning behind this approach, including everything that was tried and ruled out first.

## 1. Setup

In [ ]:
import requests
import re
import json
import time
from pathlib import Path

# ---- CONFIGURE THIS FOR ANY BOOK ----
BOOK_SLUG = "piranesidigitalproject"
INSTALL_BASE = "https://scalar.usc.edu/works"
# --------------------------------------

BOOK_URL = f"{INSTALL_BASE}/{BOOK_SLUG}"

# Slugs known to permanently fail on the source server (e.g. hidden/
# unpublished pages that trigger a server-side PHP error rather than
# ever returning valid data). Skipped outright so the crawl doesn't
# waste every round retrying something that can never succeed.
KNOWN_BROKEN_SLUGS = {
    "actually-rotated-remains-of-an-ancient-tomb-today-called-la-conocchia",  # hidden page, PHP error on source
}
KNOWN_BROKEN_URLS = {f"{BOOK_URL}/{slug}" for slug in KNOWN_BROKEN_SLUGS}
DELAY_SECONDS = 1          # pause between requests, be polite to the server
MAX_RETRIES = 3
RETRY_DELAY_SECONDS = 5

STATE_DIR = Path("crawl_state")
STATE_DIR.mkdir(exist_ok=True)

GRAPH_FILE = STATE_DIR / "merged_graph.json"           # the accumulating full export
VISITED_STRUCT_FILE = STATE_DIR / "visited_structure.json"  # composite nodes already crawled for children
VISITED_ANNO_FILE = STATE_DIR / "visited_annotations.json"  # media nodes already crawled for annotations
FAILED_FILE = STATE_DIR / "failed_requests.json"

SCALAR_TYPE = "http://www.w3.org/1999/02/22-rdf-syntax-ns#type"
COMPOSITE_TYPE = "http://scalar.usc.edu/2012/01/scalar-ns#Composite"
MEDIA_TYPE = "http://scalar.usc.edu/2012/01/scalar-ns#Media"
PAGE_TYPE = "http://scalar.usc.edu/2012/01/scalar-ns#Page"

def load_json(path, default):
    if path.exists():
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    return default

def save_json(path, data):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=1, ensure_ascii=False)

graph = load_json(GRAPH_FILE, {})
structurally_expanded = set(load_json(VISITED_STRUCT_FILE, []))
visited_annotations = set(load_json(VISITED_ANNO_FILE, []))
failed_requests = load_json(FAILED_FILE, [])

print(f"Loaded existing state: {len(graph)} nodes, "
      f"{len(structurally_expanded)} structurally-expanded, "
      f"{len(visited_annotations)} annotation-crawled, "
      f"{len(failed_requests)} previously failed.")

## 2. Core fetch helper
Every request goes through here so retries, delays, and state-saving are handled in one place.

In [ ]:
def to_rdf_node_url(url):
    """
    Convert a plain content URL (e.g. '.../piranesidigitalproject/about',
    the kind you get back from hasTarget/dcterms:references values) into
    its RDF API equivalent ('.../piranesidigitalproject/rdf/node/about').
    Leaves already-correct API URLs (containing '/rdf/node/' or ending in
    '/rdf') untouched.
    """
    stripped = url.rstrip("/")
    if "/rdf/node/" in stripped or stripped.endswith("/rdf"):
        return url
    if url.startswith(BOOK_URL):
        slug = url[len(BOOK_URL):].lstrip("/")
        return f"{BOOK_URL}/rdf/node/{slug}"
    return url

def fetch_node_rdf(slug_or_url, res=None, rec=None):
    """
    Fetch a single node's RDF-JSON, merge it into the global graph, and
    return the parsed dict (or None on failure after retries).
    slug_or_url: a bare slug (e.g. 'index'), a plain content URL (e.g.
    '.../piranesidigitalproject/about'), or a full API URL -- any of these
    are normalized to the correct '/rdf/node/...' endpoint here.
    """
    if slug_or_url.startswith("http"):
        url = to_rdf_node_url(slug_or_url)
    else:
        url = f"{BOOK_URL}/rdf/node/{slug_or_url}"

    params = []
    if res:
        params.append(f"res={res}")
    if rec is not None:
        params.append(f"rec={rec}")
    params.append("format=json")
    full_url = url + "?" + "&".join(params)

    last_reason = None
    for attempt in range(MAX_RETRIES):
        try:
            resp = requests.get(full_url, timeout=30)
            if resp.status_code == 200:
                data = json.loads(resp.text)
                graph.update(data)
                return data
            else:
                last_reason = f"HTTP {resp.status_code}"
                time.sleep(RETRY_DELAY_SECONDS)
        except requests.RequestException as e:
            last_reason = f"request error: {e}"
            time.sleep(RETRY_DELAY_SECONDS)
        except json.JSONDecodeError as e:
            last_reason = f"invalid JSON in response: {e}"
            time.sleep(RETRY_DELAY_SECONDS)

    print(f"  FAILED after {MAX_RETRIES} attempts ({last_reason}): {full_url}")
    failed_requests.append(full_url)
    return None

## 3. Structural crawl
Starts at the book root, walks every `path` relationship recursively, and discovers every real content/media item in the book — the live equivalent of the book's table of contents, at any depth.

In [ ]:
def get_seed_urls():
    """
    The book's top-level table of contents is NOT a 'path' relationship --
    it's a separate 'toc' Page node, linked from the book root, whose
    children are listed via dcterms:references. Everything BELOW that
    first level uses 'path' relationships. Cheap to call every run --
    it's a single small request and just re-merges the same book-level data.
    """
    book_data = fetch_node_rdf(f"{BOOK_URL}/rdf")
    if not book_data:
        print("Could not fetch book scope -- falling back to 'index' as sole seed.")
        return {f"{BOOK_URL}/index"}

    toc_url = None
    for uri, node in book_data.items():
        refs = node.get("http://purl.org/dc/terms/tableOfContents", [])
        if refs:
            toc_url = refs[0]["value"]
            break

    if not toc_url or toc_url not in book_data:
        print("No toc node found in book scope -- falling back to 'index' as sole seed.")
        return {f"{BOOK_URL}/index"}

    toc_node = book_data[toc_url]
    seeds = set()
    for ref in toc_node.get("http://purl.org/dc/terms/references", []):
        content_url = ref["value"].split("#")[0]
        content_url = re.sub(r"\.\d+$", "", content_url)
        seeds.add(content_url)

    print(f"Seeded {len(seeds)} top-level sections from the book's TOC.")
    return seeds

def find_pending_structural_urls():
    """
    Recomputed fresh from the graph every round: every URL that has ever
    shown up as a path-relationship TARGET (i.e. discovered as a real page
    or media item via the table-of-contents hierarchy), minus whatever
    we've already SUCCESSFULLY expanded.
    """
    pending = set()
    for uri, node in graph.items():
        if not uri.startswith("urn:scalar:path:"):
            continue
        for target in node.get("http://www.openannotation.org/ns/hasTarget", []):
            target_url = re.sub(r"\.\d+$", "", target["value"].split("#")[0])
            pending.add(target_url)
    return pending - structurally_expanded

def find_inline_media_urls():
    """
    Some media items are NEVER linked via a 'path' relationship at all --
    they're only ever embedded inline inside another page's own HTML body,
    via <a name="scalar-inline-media" ... resource="[slug]">. A path-only
    crawl will never discover these on its own. This scans every page's
    content already sitting in the graph for that exact pattern and
    returns the referenced media slugs as additional URLs to fetch.
    Excludes references that point straight at a raw uploaded file
    ('media/filename.jpg') rather than a real content-node slug, since
    those aren't separately-fetchable nodes at all.
    """
    CONTENT_KEY = "http://rdfs.org/sioc/ns#content"
    found = set()
    for uri, node in graph.items():
        for content_val in node.get(CONTENT_KEY, []):
            html = content_val.get("value", "")
            if "scalar-inline-media" not in html:
                continue
            for tag_match in re.finditer(r'<a\s[^>]*?scalar-inline-media[^>]*?>', html):
                res_match = re.search(r'resource="([^"]+)"', tag_match.group(0))
                if res_match and not res_match.group(1).startswith("media/"):
                    found.add(f"{BOOK_URL}/{res_match.group(1)}")
    return found

def crawl_structure(max_rounds=25):
    pending = (get_seed_urls() | find_pending_structural_urls() | find_inline_media_urls()) - structurally_expanded - KNOWN_BROKEN_URLS
    round_num = 0

    while pending and round_num < max_rounds:
        round_num += 1
        print(f"\n--- Round {round_num}: {len(pending)} node(s) to expand ---")

        for url in sorted(pending):
            print(f"Fetching structure: {url}")
            data = fetch_node_rdf(url, res="path", rec=1)

            if data is not None:
                structurally_expanded.add(url)   # ONLY mark done on real success
            # on failure: url is left out of structurally_expanded, so it
            # will show up in pending again on the next round/run.

            save_json(GRAPH_FILE, graph)
            save_json(VISITED_STRUCT_FILE, list(structurally_expanded))
            save_json(FAILED_FILE, failed_requests)
            time.sleep(DELAY_SECONDS)

        # Recompute from BOTH sources every round: newly-fetched pages may
        # reveal new path children AND new inline-media references.
        pending = (find_pending_structural_urls() | find_inline_media_urls()) - structurally_expanded - KNOWN_BROKEN_URLS

    if KNOWN_BROKEN_URLS:
        print(f"\n(Skipped {len(KNOWN_BROKEN_URLS)} known-permanently-broken slug(s): "
              f"{sorted(KNOWN_BROKEN_URLS)})")

    if pending:
        print(f"\nStopped after {max_rounds} rounds with {len(pending)} node(s) still "
              f"pending -- re-run this cell to continue (it will pick up where it left off).")
    else:
        print(f"\nStructural crawl complete. {len(structurally_expanded)} nodes expanded, "
              f"{len(graph)} total nodes in graph so far.")

crawl_structure()

## 4. Annotation crawl
For every Media-type node discovered above, fetch its spatial annotations the same way we validated earlier (annotations attach to the media node, not the page that displays it).

In [ ]:
def find_media_urls(graph):
    media_urls = []
    for uri, node in graph.items():
        types = [t["value"] for t in node.get(SCALAR_TYPE, [])]
        if MEDIA_TYPE in types and not uri.startswith("urn:"):
            media_urls.append(uri)
    return media_urls

media_urls = find_media_urls(graph)
pending_media = [m for m in media_urls if m not in visited_annotations]
print(f"Found {len(media_urls)} media nodes total, {len(pending_media)} still need annotation fetches.")

for i, media_url in enumerate(pending_media, start=1):
    print(f"[{i}/{len(pending_media)}] Fetching annotations: {media_url}")
    data = fetch_node_rdf(media_url, res="annotation", rec=1)

    if data is not None:
        visited_annotations.add(media_url)   # ONLY mark done on real success
    # on failure: left out of visited_annotations, so it will be retried
    # automatically the next time this cell is run.

    save_json(GRAPH_FILE, graph)
    save_json(VISITED_ANNO_FILE, list(visited_annotations))
    save_json(FAILED_FILE, failed_requests)
    time.sleep(DELAY_SECONDS)

print(f"\nAnnotation crawl complete for this pass. {len(graph)} total nodes in graph. "
      f"Re-run this cell if any media nodes were newly discovered since, or if any failed above.")

## 4a. Audit: find every inline-media reference and confirm it resolves

Scans every page's own HTML content already in the graph for `scalar-inline-media` embeds, and checks each referenced slug against what's actually been discovered as its own node. This catches the exact gap that caused a missing image on `view-of-the-entrance-to-the-tomb-chamber-of-l-arruntius-1`: media items referenced only by an inline embed, never linked via a `path` relationship, so a path-only crawl would never find them.

Safe to run at any point -- it only reads the graph, it doesn't fetch anything. Run it again after any further crawling to confirm the gap has closed.

In [ ]:
CONTENT_KEY = "http://rdfs.org/sioc/ns#content"

# slug -> list of pages that reference it inline
referenced_by = {}

for uri, node in graph.items():
    for content_val in node.get(CONTENT_KEY, []):
        html = content_val.get("value", "")
        if "scalar-inline-media" not in html:
            continue
        for tag_match in re.finditer(r'<a\s[^>]*?scalar-inline-media[^>]*?>', html):
            res_match = re.search(r'resource="([^"]+)"', tag_match.group(0))
            if not res_match:
                continue
            slug = res_match.group(1)
            if slug.startswith("media/"):
                continue  # raw uploaded file path, not a separately-fetchable node
            referenced_by.setdefault(slug, []).append(uri)

missing = {}
present = {}
for slug, pages in referenced_by.items():
    node_url = f"{BOOK_URL}/{slug}"
    if node_url in graph:
        present[slug] = pages
    else:
        missing[slug] = pages

print(f"Total distinct inline-media references found: {len(referenced_by)}")
print(f"  Already present as their own node: {len(present)}")
print(f"  MISSING (referenced, but never discovered as a node): {len(missing)}")

if missing:
    print("\nMissing media slugs and the page(s) that reference them:")
    for slug, pages in sorted(missing.items()):
        print(f"  {slug}")
        for p in pages:
            print(f"      referenced by: {p}")
else:
    print("\nEvery inline-media reference resolves to a discovered node. No gaps found.")

## 5. Retry any failures
Re-attempts anything that failed above (transient errors, momentary 503s). Safe to run more than once.

In [ ]:
if failed_requests:
    print(f"Retrying {len(failed_requests)} failed requests...")
    still_failed = []
    for url in failed_requests:
        try:
            resp = requests.get(url, timeout=30)
            if resp.status_code == 200:
                graph.update(json.loads(resp.text))
                print(f"  OK now: {url}")
            else:
                still_failed.append(url)
        except (requests.RequestException, json.JSONDecodeError):
            still_failed.append(url)
        time.sleep(DELAY_SECONDS)

    failed_requests[:] = still_failed
    save_json(GRAPH_FILE, graph)
    save_json(FAILED_FILE, failed_requests)
    print(f"\n{len(still_failed)} requests still failing after retry:")
    for url in still_failed:
        print(" ", url)
else:
    print("No failed requests to retry.")

## 6. Sanity check the merged graph
Compares the shape of what we've built against what a native Scalar export looks like.

In [ ]:
type_counts = {}
for uri, node in graph.items():
    types = [t["value"].rsplit("#", 1)[-1] for t in node.get(SCALAR_TYPE, [])]
    for t in types:
        type_counts[t] = type_counts.get(t, 0) + 1

path_nodes = sum(1 for uri in graph if uri.startswith("urn:scalar:path:"))
anno_nodes = sum(1 for uri in graph if uri.startswith("urn:scalar:anno:"))

print(f"Total nodes in merged graph: {len(graph)}")
print(f"Node type breakdown: {type_counts}")
print(f"Path relationship nodes: {path_nodes}")
print(f"Annotation relationship nodes: {anno_nodes}")
print(f"Remaining failed requests: {len(failed_requests)}")

## 7. Write and download the final export

In [ ]:
BOOK_TYPE = "http://scalar.usc.edu/2012/01/scalar-ns#Book"

# Two node types get excluded from the final import file, both confirmed
# by testing against a real destination install:
#
# 1. BOOK_TYPE -- the book-level record itself (title, background, custom
#    CSS, toc pointer). Not 'content' the way pages/media are; a real
#    native export never includes it. Importing it made the destination's
#    importer hang indefinitely (it can't add a whole Book as a page).
#
# 2. PAGE_TYPE -- Scalar auto-generates its own 'toc' page for every new
#    book, so a second one can't be imported on top of it. Confirmed by
#    the destination server's own explicit rejection: 'Invalid rdf:type
#    value.' (HTTP 400) when this node was submitted.
#
# CONSEQUENCE: excluding the toc/Page node also removes the only record
# of the book's top-level main-menu arrangement (which sections appear,
# in what order) -- that relationship isn't expressed anywhere else in
# the export. This has to be manually recreated in the destination
# book's Dashboard after import; see the methodology doc.
excluded_nodes = []
for uri, node in list(graph.items()):
    types = [t["value"] for t in node.get(SCALAR_TYPE, [])]
    if BOOK_TYPE in types or PAGE_TYPE in types:
        excluded_nodes.append((uri, types))
        del graph[uri]

if excluded_nodes:
    print(f"Excluded {len(excluded_nodes)} node(s) from the import file:")
    for uri, types in excluded_nodes:
        print(f"  {uri}  ({types})")
    print("\nBook-level metadata (title/background/custom CSS) and the top-level")
    print("main-menu arrangement must be set up manually in the destination")
    print("book's Dashboard -- neither survives this import mechanism.")
else:
    print("No Book- or Page-typed nodes found in the graph -- nothing to exclude.")

FINAL_OUTPUT = f"{BOOK_SLUG}_full_export.json"
save_json(Path(FINAL_OUTPUT), graph)
print(f"\nWrote {len(graph)} nodes to {FINAL_OUTPUT}")

from google.colab import files
files.download(FINAL_OUTPUT)

## Notes for next steps

- This file should be structurally identical to what Scalar's own `instancesof/content?rec=1&format=json` would produce for this book, if the server could generate it in one request.
- **Media files themselves are not in this JSON** — only their URLs/paths. Physical media still needs to be copied separately.
- Test this file against the destination install's Transfer tool JSON importer on a throwaway book before trusting it for the real migration.
- If any slugs are known-hidden or broken (e.g. return a PHP error rather than valid JSON), they'll show up in `failed_requests.json` inside `crawl_state/` — review that list before considering the export complete.